# Navier–Stokes Re=500 — additional reference-oriented operator baselines

**Independent companion** to `NavierStokes_Re500_CLEAN_40Run_Methods_and_Ablations.ipynb`: it **does not modify or rerun the original 40-run AWFNO/TB-AWFNO/WFNO/FNO plus ablations**.

This Colab adds **16 runs**: F-FNO (24-layer factorized spectral/weight-normalized backbone, K=20), CNO2D (multiresolution CNO blocks, initial condition plus direct-time query), U-Net (autoregressive 4-level convolutional backbone), and **UNO2D-H10** (seven-block U-shaped spectral operator from the prior Cahn–Hilliard UNO-H10 study, adapted to a 128×128 grid). All 4 N sizes **25/50/100/200**, fixed seed 1234. Architectures adapted from previously used notebooks; these are *reference-oriented adaptations*, not claims that the original authors' code is unmodified.

### Protocol disclosures
- AWFNO/other existing methods, F-FNO, CNO, U-Net receive **one initial observed field** at inference. **UNO-H10 receives 10 actual initial frames** (`t0,…,t9`) as specifically requested. UNO is trained on teacher-forced ten-frame-to-next-frame pairs; this **does not reproduce the original UNO-H10 40-step differentiable unrolling**. We preserve the seven-block spectral UNO architecture and ten-frame input, but label the optimization adaptation. At test time, after those first ten observed frames it is **fully free-running**.
- F-FNO is a factorized Fourier operator with **shared** 1D Fourier weights across 24 residual FourierFlow-style layers, output K=20, rolled autoregressively; CNO is direct-time (not recursive); U-Net is 1-step recursive. Different target/optimization protocols and different numbers of observed frames preclude claiming a strict apples-to-apples architecture-only benchmark.
- Fixed **verified** `(300,128,128,101)` dataset, source hash, train 0..199, val 200..229, test 230..299, physics `(2π)^2`, dt=0.0025, no test-based selection. Training-only normalization and validation checkpoint.
- Report original full-time absolute physical L2, final-time physical L2, a **common [t9,t100] forecast window L2**, relative H1 gradient seminorm, final-time spectral-band/sharp-gradient errors, train/inference time, parameter counts, and side-by-side truth/prediction fields. UNO full-time L2 contains 10 observed (zero-error) frames and **must not be interpreted as equal-input performance**.
- **Run all** writes recoverable checkpoints to a **separate** Drive results folder; the full 16-run test set stays sealed until all 16 fits have finished. Rerun the training cell after a session timeout. The original publication results are merged when the original 40-run notebook has finished.

**Do not claim numerical results until the GPU runs and outputs have been inspected.**


In [ ]:
#@title 1. Mount Drive and use the ORIGINAL verified Re=500 dataset
from google.colab import drive
from pathlib import Path
import torch,os,json,sys,subprocess
from IPython.display import display,Image
import pandas as pd
import numpy as np
drive.mount('/content/drive')
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU'
DATASET=Path('/content/drive/MyDrive/Kolmogorov_NS_Re500_start0.25_horizon0.25_dt0.0025_v1/kolmogorov_dataset.pt')
ORIGINAL_ROOT=Path('/content/drive/MyDrive/AWFNO_NS_Re500_101_CLEAN_FULL_v1')
NEW_ROOT=Path('/content/drive/MyDrive/AWFNO_NS_Re500_101_NEW_REFERENCE_BASELINES_H10_v2')
assert DATASET.is_file(),f'Original dataset missing: {DATASET}'
NEW_ROOT.mkdir(parents=True,exist_ok=True)
MAX_EPOCHS=45 #@param {type:'integer'}
BATCH_SIZE=8 #@param {type:'integer'}
SESSION_MINUTES=180 #@param {type:'integer'}
SEED=1234 #@param {type:'integer'}
print('GPU',torch.cuda.get_device_name(0),'Dataset',DATASET,'Output',NEW_ROOT)


In [ ]:
#@title 2. Restore the validated original metrics/dataset loader and additional model driver
import base64,gzip,hashlib,py_compile
payloads={'ns_base_original.py':'...','ns_new_baselines.py':'H4sIABco22oC/919/Vfjtrbo7/krfOg6KzY4IQkMbcNx16MDdGYdJswamHOmNy/XyyRO8ODYqe0wSVn872/vLcmWbDkJtL3v3jst4A9pS9ra2t+Sv/vb4TJNDu+C6NCPHo3FOruPo6PG3t7e2WRiJP7UT/xo7LfiJPCjzJ8Yl63LwbVtvMVfn1sDPzO8aGJ8Hlwbd1Dwfu4lD6mRxcajFwYTD2sMbg4/+W86HSP1s+Wi3Wic+0nwCC+8ibfIgjhKjXhqZPe+sUz9pJka3XPj52Uy85OUYPfOjbfefdR6F4Rh4CW6XjWC+SL053DtMYAm9M9I8WkwDaCpO2/8cBdHvm1cxkuolFyG8bdWmq1DX4yIBmNhiw0YTetdt2OoUG0jjgzPmC7DcG0sYAzxJBgb3d4P/9kzZkkwaRs3AHvsG14yvg8yf5wtE7/hZVkS3C0RgBGkRhDRSL3xOJ4vvGgdRDMjijP/Lo4fTvFV6jO88IF4iW8Mrm8b49AL5jAQwOwdICqaQ+M0sng6DcaBFxqAi1kQwYW3hDlM0qaRJV4QYQPpOAkWWdrGaW1Mk3huuO50id1zXRxknOAsQjdYow3xKJktvCQFpI3jxdo27r30PgzubONritiYe9k94CS1jQSQFs9tI4Mu2ka6hkezMWtoAYWgjmjlI9wK8NFyvlgbXmpEC/EoiwF1rCJdimpRJD1sR5F4Pl1GY+wyjjo1LtVCbUB6mIqi3/xgdp+5UZzM5WJUpg2E6omC53ANlJr3MnXv4N4t0Jsa+KDRGHxxBr86eN0efGkMbvnlbeOcX57D5Rd++aWxcpFInGjR9gBfM98cfLGNSbZe+PhsGsZednJs7UPJdaXkrzUl/3Xx6eb99cDZi9JWgmusVawNfwXz38LWwyDy09Z9t9N67O01Plzcvrs+v3HMvUug+95x65+9zp5t7MGKaZ0HCZDtLUwjPvkMC6J19qnLrq9757go9qzGzfv/uLhh46LLxtvrweX7X5ynhkEw3X+/P799t9c/ObbZ/dXZr9DRvX5PPPhwfX4B990Tfn959vb2+tNe/9gWIC4vi1q80M27s08X7uX150/vLz65/754/8u7273+bbL0eQH2yB1cf/rAn+fQeKX3g/e37i9n7wd7/U67y6tdf751b24vPmJTHazyVurykU23b9+dDQYXV+6Hz1e37z9eASx41WPvPl3cuD9fXb/9JxXn9QcXb/8pPaVnn28u3J/Pbt++K3Xx8+DiFl7AW97MDQNOz68u/kUPjvk9q7o3S+LlYi+v/fHs/Pz94BdCLLwdB8l4GXoJFeCzgRAFnjvt79/Ye+/e38Ddr9DE4Bcs0aXR87kZdrv29/YR/fe93e2O7D1s+uwKZjwf2KUXAntoPDe+Mz4D45oC/wOuG/mH4xj/AN+IkwmsmswHlpyzTOCEw47d218E1n/2rHZj4k8NXJcgMX73J0T87vgeGJIfpubKBUmy9FN7zS+sfsOAfwkIkySipdgWkN2p7yFXS83meLFsWtCK0T8eNeAf8M80Nf49uIJueYkJ3IFdcWjYBdcFbpm5rgkrZmoDp87BAZdbZtIdyClX4iccBv5Ll9AX02rnsDZAuQu81EEasPLqwbQMuwCN/6QX1EsrH9ml708u4+QbyEcc3Id4sgx93eCKruIo87tJMC9upt4YuGNxH7mhtwZxXDwp9VKCk8QLGKbTaXfYw03IUQYuGjH+YXTVUYMYA+L6F87+RZLEibmXl50v0wwF4gLoLQOlor1XwORFHGM4KkYWJzCxE3+FgpgxVwHLUhuFiQOcQG38Dd1jtRzH6BhAlj493ueYUiriJOtr5r1uGd0tQFjBtrdY+NHEzMmWdcoWbVQp0doABcjinM2OyWdpY3HlFVFB1P7kX32GTixCb+yXSFeaSTbgfyjjrZSj8QPI9xPQr4JsbaqgLIVO2/lcQo0b/7cl1vFCc59PXSOn8ylfA2wNr6RJ5QxDAmeupPVDc4D852YBIjDxwiss0zt/1Wr6FkxAN8pvQVnz003LazrdbYXttphoiNQFQFgQZSZdl95Tn/h7ui69R4157KXAsKZQSmYvyjwBFTql0RZDdMojlZmJox22ZuhOLbPRMhxGPWWCcKdM7xeEYRv8AYcsL//vjFXfMH+2jQEUG4Di+9YibRe0elCmJQMCbBUSVr1Je7Fu5/VXufQCzK1QOs2XmW+CMDqyja5tgMwbx0DAs2W8TKV5u8MmbMO1jQgajtZYOwfVTu+9hd+QOnkL/CuF4c0NL4yhY0oPppmL9ZmSO51m7QR+mQU4m2auBb0hFO+BqnsfS8yTRJUM43c/iVN17ll/lUcF6ZUmfVW6XxuHh0bPOAB8qNNJai7rf5tuSu/9xwB4z6rNLuQplxY62D2gSGDn50FkFvRuyw0XNThxFYN9DPxvrpe6aKSF/koddolwhp3RsA96BvxftAw3I/0kV1EsaisARnlf/CBKl3O1C3t3wWptB/G69dNdvFrvqThabQBr6/SJtQ6NmSAvjsmClAKipYrQw0YBwU601hNXeXXqqXhVouLVDlTc20LFqz+NimuoNlrXUvHqj1LxSkfFq3oqXr2OirtVKl69gIpX+to7UfGqnoprwGqpeLWNilc7UvGKqHilJy4dFUuNYBNKkwfqQiqrI9JLWVD0mKwoCYo6LWeTMJPVHkmmM3IqC0cNJElFQnfBuXv2abNKxPoE/Z4Gs930FVlVYfWGsh9hZKmKlK4kM1pHJR0mV0A1NbiBL1UpqRlQ6S6Ow1KPJBfDSKIA2dB1NhmzeG8bzMNT6m3iz4I0A9TfLadTwJW6VqQWSgtFetNeRikox/7vQESWWgqwnyL4KHO41a5lxzcZwDEW4TI1VuR9XUvwjVwZUXsOZskiib/CyHNL5Y3NleCKidJQ65bojan4H73Em/uAiyvoskQpaL65henWK9lsC1GtBKVqy3CmNF+A4cH7yf9wDturM0e4JYRE3F55j9h1Nt1utZW8P3bl1Qy4mUOuPJXAqp6qUWkiraqoKuFQWHB582Wcy9YUW8iEZwXwsNLlehOpUjRf1TrbQFnMTsk20hgRlaXLvVhlxGjMKU3dy2Ll66u/yPqoToiGTBVepJYeNWoMXhBFFWtXqZmvNE633d4P1bVm66tQ2QpqCj8ooGYDqFo5JLG9FNmItDqDaeVtO0Lvx98c42iLt6eK3r2LFVIgSFuw0sBIG6yhw4k/9ims9FRpiUyn5/ZezToCJfQ+SAHvgI6JMVtijAnuJ34Y3PkJAmAxInzohYnvTdaGvxqHywlGltYSHG++cP3Iuwt9lKkg7mzjDiYSLsdoOmLkhyljGQ87CQwCq0gxbCbBIuZw1IMW09gA9ooBJWgfGUeaGZ4BQwt9AyjNj8YhultnGPmJUb6j9M38VdaWwL39fH5mTGKQT1GcFZEt6tNbpiC+88KpwRQ1nH4Ymw9GbluS0yCmGecULZkatdZlkYryJLCX7aoyzBFWEUslspD0GDfXWxiOzEpjjLNaEqlW6mylWTIMVGkukROYfTXS36wyaFlE+6sFyFWTG/toJLEflTHolEENryhU2rFXZuCVfhfKAbRmy70eCYutpglZypPXrMAR0yihzCCOfEVQE79DLijJnH7J9Mpr02vUP3UCTUUNdgj1alFbq+qil1MUsLa7YISOS6Ef8nLuoOMCQlJAL3PH4tUL3HO8LtdM+V1VBsiF8lbUUiAkg0cK3DKBceV7D2sawk5uUcTmJXQH1IRFHAIxqFRU8ttg+47ZM/aVUQA+S09KkgeFvAPm3ngJPyX1FUh0FoGNmkQosEtMgAqQ9BPBEnsTiRa4QCot0cVLR6nMga1OyX/N+CSq/DmMxw+v8kXDjOTuEsVqrz4V06kUU5+gWnAXST19Ac3DinuMw2VBqm/hQa/kUNZ2t77L+O/BB9yG1FEHlvfCmwBbmznd/NJl8yNHRfVaFzFlYf2ZZZ3/Z3w7gJfQZ7k3lojb3UWbwhqb1m3Bd6pM5aXBDWkRlAZllqcBVomlkNlVMM0+ApMHXWMHisvZYOGPU2bJYCPB9RZRwMo5OX4Rj4TlOvcnAZqeTrEKdqOXolW7XIEoRaVrmdo1rzjZVyznl9N3Xbf+IvJ+Ie3ItFGZA6QWmVg++WkwWXrhzpRSJgyJmbyOjXTr8Vy0VVy9GJM7zHTv/1MXilXd3car/hifKhrq/bUN1TBELn1354SMKIVKIFOLqhSo5SR8muJdfWmJx24vLFPLDsUlhGtL02pFBVi8LK3JgZ+9ZjFGy7l7hys5fe3CZLU3uy32q36lCifRs4mtbhbUkYtBWPWOlhfyRAZPThZ4S47wIlnuz3aI1/it32rd1hxb7nwZZsEiDMj7Wa6myV8bye0lvq4xKbOtXNqN/PGDpoqc91byrd9FZZe6JilOduEz02fwRfYiDb6gy2jwKzo+qMDfDbO3v5971/BtZ1v6ECYH++RCSg0PLN/HIA3uQt8A8xjzbSmZd6/sNJUmRVz+Of77XaMFf9S5H4JuxzOTNDSD8cRG4ZIZgxRK8mQ1TKIS1V+TTFWGV5tjhHNJIC0wKKu9rHHgTXwVPKwodCmgTzvxH6E7/sQsd2HY7Y+sDSCYA3YLjH6rKwPRIARj4HqcaDo9pMojY9+BuahMBpI7zQTRPcwXIctirVangILDo8oAcygVi5eBNPb3DVPK3ApKjHRTa42C20qex2WHQpRr2wDqTdaUJV4KICFtMaFfMUBO7JxwmTxwND6QBatRA0LVv8UqONDO+RBmVF1bXY1vQDUQypyCT9hL4yt6+6ZuCQ2Dkb1jQZybLYWJKraC5KU2wGN8ftcYiYaOdgyPpA/BwiVPappbW/8NMP2HsKys0KG8BP8qZKuLdiPCeef+XExrWeAW7CiLdTdU7ljqvwFZM8UqS1+BZq7470aoIUi0cCdi3VSSFME/A2HUyKuRxjXRGhxUxi6gj+xG/bBrChWqr8499LIQKYpFEoguCsTaWOmys3OgVK1FUSprUzRLFKoPZb0ieIWDESosQChGiMSNrYnKXfa/JWAqY2uxt7hFavDr6+NeMMJCaS7FvGylqzwCtjEAhqqD7EJAIbR1e4Ea/FdnkKQY7WoRTQgGwDVBuTVRQejORWW1DI9mT+MwjL+lFN/N9whKOyLVvY1gDnnjLFz3S6Cw9iT+FhlkBBseWkxQekVbFQ223gw2AkxiY81RLyshO1k10gwvL4KrS4kxvhCpyl6NfoXf5O2oKsRQAVlBPHqxqrAY5WJ8lVGCDq6WPaoGZYt1uFJyRHM8bDFkWdqhFItgCDq20iFB0pZ+Krj0k6aiUQ9YN7IO719nZFWbkh0oXDuHJtrFWmw0aN8X7WNzx/ES/QjCfcgwjfNOr2nXrPmDbRxTLLyr8so81/7vonR14nlv2Ht5+1g39+ngZkd04v4ZYY+X+Hpkty9a5tw1Ut3VV3AZZHDuA8xbuTzzoDQkm68QJcxZWUJxSeRwoI7BtxdWKZ6jDdDzC5ZAv6+pncLCv2htbomkwcaWtB7mzUCjOPI3wqzZdaQVqQS2bOMxfrjR1VkTFdgYtNocCapGgzQphyKgUPeKBRjkm1J+od47xehHCXva5eH+coFZB/auWNg80P89aHidx1l2OCNz+ktSr3H7bMmLq9mNLMkeUtVTXRW+UXmkbOrkxUF5Pd7mlCWtxVtOgqyqnYB+nbLdxQygvM1T2ms1xOGwZBR2dZxf/ZBfdU/45eh/lvtW1p9ebCCq0u1NwZ1BeucUY2+pptYRN91XAOjKAHqvANCTARzVAag1GRdxHHIUequPcAPsqVeOZsVZFvrcuFS7oY3ysm6Im+O8T3V9WC4wLvjimeT8lLZNLeLU1zFWpVvHZVTJrBPUqTQD6oSrqq1u//HGj8oT/V/ZeK9Mpv+VjSsk3nl949s8cn+QFfSKOEtpto52X5YKkN4r17YCpPtKDqMA6Yx24XObUvAXIgufqzAKtK46p93X5MW/Nsf4FY6ZklG3ITtYnxzc3eb4EGEPkfErZFW/Yn/yN5u8G6saqxU5t9ZtULBr+XXuO7ANkzFcW7h9LeyoHy3nlOKvrprfg4UpM2lbXXJWXZI69oZV0Q2gZFKTbW8yXwXF6uoNdt6wJqlVotSKfS20R7FdhxHx58H1S/JIN9m50Nsu/e7xfVNd/rf30tRQqZEiJbpq7OU5uKXCij5eYpMBJUhhKbysvu0Vb3u6MxK68iEJXV2JnlxChjHGHRqOYXbbHQNjq/BnvzJey8KQa6f9pqiHa1+/kaDWcKxgRvOaT1HNi15tXiPPwu9u3FXHBrvPiRxPBIvMfRqJOLuKkz/tv7C2tNX7i9qiy/+DDC8YA+D7eJLTP98lLbIN1rgZoXY7rbKpOd/HnG/HN6S6G3btIkE6AzqWDqmPLqW22LJ2OQUX1Ax2Ffub0qYLlmJXJfCiek+q3uPVe9rqEvXOvZWbLwDcgL7iQqnVG1G2iK10EJ9oKvfKlbujYg+90sXqhnboqLwMf5K7hOk/0gKU3vVevp3sA20rfjLlVSKDt55xv5ePx+/h2X2TeZCyhKE9DbAwmAcZACs6a0udA1Aok3QVn7IlEKCE5v4ISrd+AlgSohWkWdVNbVLixeXlrREvULzRUYKJP/dA5MH/fAvUISf5k+NT4+zDR9rzFqS0GUs9hKFyBEMPesm3WInd8Zy4K4cvbDp6YZWrOS9iZjI2al70Nh4vInOHFx/MwIYlziVQOKty2xvV7QUr8ZmqX39H+FVlWGHXdduGa4bSktda/y8Yy+YGNg+mt30wCmfOD3jo6U54KFGbU7++qq6egtJ1O3U+xsBIvwHHuV5sOzrqBdrVC5PjFZOlrEvZVYVJ2Vv4GqXpf6iEW8kJ2n/ChrEdqWj3zWLqfrDd9opdE8OPE7KHX3l82Qu2jKlMmEwB9bSObuleep+boGWvZxRHIe1UD7L167eawexWLZ/avUMvM210miu0py7/l7dVm+IiMCUyqPMHlYIF5oqyxbOqcpVD6lfZe/6O5329j0BzjsY+D8ZpwloVzuJNp9A2TaJOUXkN0ygtW4VRgsRnk1F6vuuwiyyEfOgyYyhqFyitArhsz/xwqWEoKymgc907fxeksFDXlSW6t7fHX7WyuAWDb5GHhlViO/Nx/33izxIfFNFHP1y38WTnOhmzcpW0l7V6+0dP6qkc0sOO7YBSpeNNxKkdecF7NsgSvNKptJrTfygpWxSvnPxTWSuiZOX42nKoKuIHIWK46vstVgQ/MlHStq9Fkk1+zq10cI4/v/Mp0tjH83HxeFzUw/kJO7KyvbhnETnmK4gW7UUAf9WlZioTarTUCcZ8DKX4oWEOvgCQ8y86tWm9W5PrUpPrrU3+WtskH6UttQ2tzv30HgNlJUFbKmszdx7GdveCr7rdcaWjmBZtWDzlmI0mrALl0ExlregCAKzAelMBmNbNELCAHkJJ//VWQYoH6BXja3sp2i6mOAH8qPf646OEkpzEc5dOYTel12qgsjYSyRevC8tmRjyBPyid7jTmhxKwg2fEij/AVB7GTFS/Aa/UUWoVBXkl+eAtLEoPi0dCde3lZ+vCRekM1LHQYI83FTrihX7YUIi6fIXdKKteapiwA6wWxNGPJ+yHGAjFEYqrnHXV6QFX3a3NQBNjwNPJMfthwLsj+Wp7M72tzUAT4yPbOOqxHwa8N5KvtjdztLWZI10zRyP5anszxzs1U0Ha8Ui+2t7Mm23NUHCqpyODNyP5antTJzs11c0PNaMDn+gXa+VkJF9tbW867ioLsidWAcuw0C2I6binVDkuqtQHyjh7UPP7+MM8C/r4jxwXxeJjK3tti7bkg6NEU1vOi+JBNnZIb97rCifkJ/ZKQzFZFVIwTJbbbG0ZjdT5p8GX59XT4NdnucsC5LOcElNgTXTqb46Oa/8BTApO/qQBCx3U+TcreBbF632Yf3bQkzesP/tIVlBIbnE9ntMznX4gl+hUinRMfG5pzxtqyMdDd8RorliVjmAIcgvj3Cdy1UWVryMYlFIo93xc9bBQVzBLpdBRXugIC/W0hY7zQsdY6Ejb3HEpjglPyMDoVoOX8PRNDvKNyYpqhvmmAvINgexoQZ7kIE9MVlQwt0oxFeiJzWatFir+2X4Gqy7+CszOVCmBJuzEsoqIbIuHZBuN74xWq2Vc8+/JtFgYqvh6Uou+DJRRfEF8wccWn1PCZDi0GYBuMzoULh7HIcLjduXZ5Ks39qPsoxckqck/ZVPrffxsBxM5uEWd/+x8PmWxxklK36BJvSTx1ibc2sx9DhqRJUGEVcwBWn2OFbSlBAxr3xzcitXFasz8LMj8uejGV5nhO6Le8OvhIas5Os2cr3+XoNCWG1bw85AcnNJGV96F5bDdbtvZSE27408PuupzYZajm4Kb3joMgpHNPw3VZxsmYB4iPLovM8hCZ59r8h69IMQoijFZJnTsn/gOUxyBpc4YwVlm+Lhvgk0pvqDY0hQaZVABOp4JMQ38cMLggl2Om6Anp3B1l7Kk44ZsR7Pvo1BZ/B7XIvHBCqXYj02H/hHRZMkS5IEZxXDrje/9RJxOaLXFGF9KLcZfQC6d19NLRyaYjnXwo3xOM25EAFxtIZ6iIKOX1o99ohnBHXp2xy4xBrtcp0pjnMiKcyNuPAyf/K9cpwfd7St1lfFIHLPQTbY4gUFz83QZhiaqSaAl2cyXlFm8KcuqOtcyW7vm+UaTuffgu4zXmuwPHw0oS+zecZrFF7GaAkXFqdfsK1dWpZL64ay8YuWQkLr64jNbec0i27u+Cv8al1RHcijyzGOWeGzLQKp+LIYKjqSxF4Y5ksA+CG12Y4tM9e9AowpSKV2NMT7jZ7tr40x9MUyMlC8yHD487bGnbK+dSMYDsptmxjIqoHDes2UqqEcmHgkv/E0b0agUV9LbdsAnryzpAz37CNb8FkA1tCB3xcY/XWAg2l5tmSCWOeeSxsv1Xj5DPGA9973ITjNB3GDpOFSM3TmrdhabvGgUR+ysHHSm0VE/IqZPhdeOud6pdAvbtA6h0a3oKJa+UyhnJiGm3x0VgBiO+r2RZUu6Gm3CWznmqtSiYAD2mmOJSz5XnLWr0jJx0zK+bHbqEsvO5cijWm0U06Z1it+YGQKHg5E7PPFROg83iPhn9yjNw5QYJWZ7LIqdix0beS6yW6nF8m6sSeouHGSti/7ioCg2KuWdL6MHhYEOcxZbbEMnWCPCYqe0aQoETOCFjkmAOOtVCLKY/fIcF3tpkix1hhzUqLwpS7uYq3F+5CAOh6HdH/4g4w/5v93rWH1t/vXvTi0PY7m32loZCAcH05R6DH7rQV+OxitSVH9HqsWaI+uUjQGf4INWlz0vbSHVc6tX4EM69IqkoR4VDM5r0KEMU1OuPJSceVagcdVUJrJ+tzOS2CowVbu3A6lJ5MaBArIf+g+gZpVpBDXAbXjr4kS/nIaE12gH6tCXEQiR2F8xnG5/ZP+usDw9xmvZahGN+sWPlvgFxgkVa5H45STVwmPOA7I7cHd+4Kd9EOjINMc+hqhkw6G9bT1269FI2+v5sQtOodK5YfCAh7VRX7he95DrdS+dEAmPAqTcrkDmyydMniIqp5sWUOf9JDO4q89xTM7c3YUFeLG57spluL0un8nA0ypED8z1PlD9AdG/EK7t8WIJv1mUpjQMlEWiLq2v6rJS6ontrxiHisZ4DinmteB3AO3Ss4w+7ceEKUy4zz4ag+Gi7Ec37JlIIXZWnA6In4u+S/GMQ99Y3K/TYIyHDSy8sc8o76pHdJN6cD3MfrSzbqczMii7BqSrbcDsMmmbtoX9+Q1tmjgC6wgI40e78vnb02/Dzsj5hgmljkie9pPEob7BWvqxPyqHzKBWCztd+5ahKf0N28HEXoC3Dz82ReNMEDutnoXf3cVP76oITX9LMvP8dp9XTH/b/zakDAZY0TyWZwmECgS5iR96+NFK975bweh3RRgZ6HAJSxTdOqBjTAoET/A72gQhPTVmiTfBj2Gn5ExAgx+/RhAG2XLiG/OAf1Zr4WBDmqGD6cbIqfImtwexBfOxmguNhZMYbLVHG1DERtuzWsXT4iEs8x4gz7L/b3k96oB0dUC6ORACMVvZs7VDXVu0Mut0ubKX/J4bv34kpnO22p+tDmbr/dlaTGnPRoDSlE7y0svV/nJ1sFzvL+tLZzKVAg6JJjNGlJwmOZKe+ICbxAKlmXdTfx40+4KGTD/azxjtgHmA/ehahxgdhy7M4c6cVN/bXb911LHyiHJzio7FDY34ESwB6KMMeMKfqcCehWE4ndEHLbxklgrWG6lf4kWRYeJBeSA3HP5hanvCPBsuML7emxP2zeiLLx8v3t5egG18dnuGX3UW/ebyjYMfuOQ3c0A99/2Jgy238comRuGyrA6HGbcCgr+Ix/cpK4s51+zeRncqKufshbizQeFzlRr5vYAHzMnFw//WrEB+K1sL9Ka4zwcTjJOYGWruOIpFC8pDUTZM2Oswsb/xkX6b2N+CaBJ/c/lGwW7+ecvUD32Szk6z8Aq7KDVcjzNhN+cxVz33eu7PPDqCJ3VvmwIKpmO76BF08fAiPBQ7B3srcf9kyXGiil0+r3a8yCqWFBWnSbP96RRhPnKL1ZbGr9hYNNNgZMGIAVTqFBs7UKKRVOUtWvbMj1jgk5f6RdyDmJx70RIGjW2b+OsA5Az8O9qnPlmoZobFF6WiNPMXqdM5zeIMDKFO+9Qbj5dgKOFfN2XOOadDZWGglMHuEmtBks5iF4+ZkD6DSyIuA12hYu6xYVma8RcTCjTOyw0JSJ9+H0hVRtKhIUECijDHyTCQDD4BbKSGK1VD8TfgU1TlN6xC0MDULhXpVosU9ru9dnb3RRT9BuHjbHAyNUqKFQkrrltx7SnXpop3VcUqjNPUuWzPgbLxkklXNfmRHOLMdk+nqDr6Jha1+swDc4nSD5RhyujkodjmAHkOFmUufWrGeGL9eW4W4AnSvsm9qzQd1mFpLQAx4vdJKPotjRsJ8MDpyhR44CiACm82Uu0B/4gYttie+DB396Du7etrwLAluD85pS4Zgn5lovvJkSm4cmyMBO9vZXh9rfGwIG8IW/f5F8pSs8aIgCYWizYuOuZ3zCiju8+ftefL0DVLzR5KfVIVZqzCF0IUtZdZAPrmOAwWtKjpxG3XrHbM7rY7mvNmKuSDQHYkn0JZA1RoCEhwHGRQ6HvagfnkWWbE1DgJ1TCzfKsEENChwlxZ7VzzD+nkvBq5DzqBo9cL5EZImrFk+MMmSJO0ecjKHQJCnqJnYtZPuXR/dp9ISbj30nsu401oxHpu8k6lHswzLT93fO+PHxaIYZN1lakGJJSYELoD+Ua/3OKebXcWlrwNStIi9dEdAgO3sSk2umy+cBjUw2bRUBseN9l7xiyhNyZpPtS0w6iHWnDpsYW9CeaYSOywGZVeUacc1jUt9WN/ndIgHO14HGloGx0PpXE7yvgdjgWmWwEy9LBo7G4SzfhimgFFwh1rHsZFVie9R/0dpHg8pzLivd4YXwNVRFSrqLKxxng58aRe4K3aFRckjUkfI5BKBKmbR2jhJe0NoQxzG+bWEnPfTnz6yr2poYFF1syN4yBzmXP/s01rIFJloIgx8RAdP9/SOiXFTX7W63Ts3lFHKBLYplO7/k7Zm/b8YRIk6J1AVsI2SvhgqWRu/CBxhi3rdIJ8RIxShKrxnMtlmLW/pnHUFG59LNmmBpBd8/WNJdrA7SapSe/x83gufnxOeB2EFsCUfyzpls4eSE3mkmesgudFuEkccwuOKiJzoPnMWYVVKJCOJronuRSFBscJhVZk+2zizf+tY/eyNs4+gDjxx95a6OVWYyfvNmepbNSXCJ5lFLCh9jq2CHFvdu0JMNWgMQsKNzb6Y0XtSl6DUjn184Jq/ohUijQCcpGX7R2dYbMpCMRWXKVWnN3jh0pzK84lTQTEF/FApuQ0g2jatE4lXshecxaIq/hU8Lnh6FSo96D9PWJJEP5M/iHXnjj6lS1ml8oU1F4o64KKkJJNKmXPvQXomWOPGWXjxbLJKSd10RnDtjepu0iyYZNx2ebob7g6+9XMv+aHIJ0jbvyJUfTReKI2ZW2B0TCtLEm8YBP0pjliGoSuQC6cmpJyL+MfC9F9c8RmAh/gBb/n8yCeurxsQ5ZgfHryMnQL9cVU4Qt+3RSzhs/oCp7Q/FFHmKySwHMprMgfLJlLKBx8IYTSXAhhoVxMYaGiRFGgEEnNUXX6uPRpjmTNlLKzpE9dAv1jwWZfkkBpRUap8KzSsSpZsiaLUnyGoJomK5TSRaGNUrUheghc5nXvAeCqjh1MDV6Sz5wj6Rf9O+DmD5U6sJAOpJNyicP58wC0sObw08XN5w8Xo1ypNQYOKHmMpIwnibIEASu1L9/fylWhZp99RDl1mFpIN/ysSSJuq29TPizY76wEXrqTeCq/bpaGMGVyzuBa8SGj4SdZE4ZaxJ2Y8fxE1/AsNzW4Vf1U4oZiVGzx0LHHCz+Zsh7jrqrcUcAwkjsKJMwcdO2SA+tADvEx3RdMW1DWnVe7Zwp3Wpmf01ClbG5mTkTFx04ESTl0gbJasOfIi5rKOmGIdei4Brr+u+pHc5xO/sopu+1K3/YWDnJnW3Sf6VaVcWtVSM2/7o7CC/MCuCmvbu/lik4P9ZEkGKeSb18pmOORYY+2QHmROR+SjxhWK6mJCoblMvi4ObIqlqkA+w9cx63csQkoyrzq+ieeLqrIHJ1+a4tzdv700H/MPQ48TDUOgQWa7JMkD/ZjYejLZk8bc8pApj7rGAuX0Mq5yyV+w+IRYUER5Q2FIiZGzck2Vr50nGIRSQwyx8NmWsknJa+CN4Wd50qmlKUsh21yYR2N75M4AiBmyc2DRP94oOEnLWIZL1xz5a/wShz44uP123cVHswY9/PhU2mJPjNu5Dzl+Oy338yeKwy3YLzQjaue8yQwzYoTET7hb3bvgZZTkOImcIgRaJ0cGSedfrs7fQZitxnXcp7oz7OGJzGcFFrVDuwI3wAl/qQGEfBpOZilmybOb1OMjjA2/5OjPINuLzM/3T8pzc6f7e8gXEneDhk1muGhOkOwfirHSDYR0dmnq1+Nm9vrjyVKUvSAZ1TlFQUDDfU/E3dFp5rDm4sbjEiNDOTjxt1yMvMzw1/de0vAxeQUFTj/Lo4fjLEXAUGCSgGyFRYffmVvCZzf0h0onn/zGpGX41qcO8F1+k8wDmiz8AJSUTmjX9LukXMqKGtKZm5FhS/aZKX+ciH5WuHIReE20ch0vDAG+kVsueNl8ujnxM7q2FLUq1muJvwXoo6Sr1MUG8NV7tmRgmiPcZIF4yBbu532AlRxO++f3bGnTZWW/yVNIO5i7nU6vD+TJTtzaReuXfYjFrMru/QkjxzOlZM7VTC8YouRUIViOMxRReQaf3OYl7ImxMrgu+h1dbT+V7sc2W0UOTTcdeLUaOg2qeR4ZpuXoGbu6PT0HN4/nV6nzrvCCMzsbCfCbgEwvkt9oKKJG0R4IMsUu5s6ZrdTkwFcAVCkQ7lixw0hk/ZPP/jr1BzKnbTLPbKLZD9bamdkN5uGhQ5LEdFqNotGGcfnBooL1MPVMsVbr1U5chCMTeccYCJcv5idkJt7NY7lhqzvSQtEaEpMHeUrUiisTDct8hCqtYXWVK7OdNnt9aupSBzU5iSlCmCOXR8z4VJHLFe74MOAysTctKyKPtZnAsjuLSogmVawIodNDWWBAf6kOBSbbN85OUvxuds7ps+GuDw9gEhOJH6UvYdNlv5HwX+XfYqHEMRT5cg7VdQuclKb6vki0PtoFvpEZXLx3M9YLg8P2ZlgdFKjm/JjeFx4LC1H1ge2IJvPQ7YKRgWr9rIYMOaih5l8yzYgTaQjSep/oQU1dtOz/fkCOPwYdz+ZSqwK4EsZ5P4YGJY/IYenmuoy3MhLa5moxgOf0xEoXPcS0emKMrOKDvC7ef8fFzd0y3cPwrMPF7fvrs9vRnwE0rLJKNRB0BahJ8J3wXwRY4gdNDwvNeD/xURQZuoMJbe++RH6Zn4FcoW/YHHWRAmsquu/cGJ9pSQCaJy2Qe8KMEiBZWCUZiRmFjtX6HiLSRu3TpF33cRXVjuFMbGvIiFb5pMCfJfhCRprZ7E7Th9NJS4pIStPg2lDqaZNp42UHbiKsvuvs6v352e3IBON27Ofr9D3hUyaugMWE14T1p+NnBMzbQ6DR2mzTIAi+Mp3B2ICw/jBn1CqjggyITw1wMTzJdCXuDN2hW9bnaBckXwM4iVPjBJowmNzLs7d24ubW/fj1dlADQ+JOgVgNB4kQsoLyITyNwdblZIKy37w5hUhgG95RRKac7+4rP3JvEK0YxfjYUQ90gTjekcd+6jTKRxzOSbkPRwUjysh9lSNnKGQqETN9JEz6IzwUWwKn1mntMEwkjwcLHIhBSA2yKiXhiU4DVU4F7VZhCssxwEcYAHJD1+NwkEZec2JUNxpTbCCtVE8aY7K8Q2+D4fUX14cryliMHFyAHJ04IUul1PpBcykn4He5T3AgOZ4WAGLVMqO0U0u3p2sMQlHNbuSGnU2mFT1xV5K2qzkcn2dx1TcVGz0onFJgwAUcCVLQg8daMvwAjwHaQwEJZ/iw97+fq9j1WCf9a3TLvx8mxymp/ddZ0u+tMSVUZljwWKhcOChSJQcWeuJzRJHlnUbA+E1Ii63rrLE5goypWMKNbeiH4tCSLjlMvBM1lQlMZpDngQ5bHII473sNtZVrurHchdfqD5vhluMagtYPlRhFZRxVmMUSI2Liioeeb0q9Fq8CVd6HdbLmdRyT++7w5qc7u1d3wpXn8e9CXCxEZFbN+73HYcveLv6Eg1LaVc8L3j4fUcDmphhAWK2WLrz4M7Bx5rS+/tPYBsfNN3mAXLO/iMeZoBhAX9tz0lhpfUqggEieTZDzxTFDSZ5nKCkeNX6h2wSws3S1x/o44Eg499YGudgxRs0ZYK8cAQ9Bc9lV1Cw0U02bT59He4x5rw3Ql8R3HJFFO9Re2IOoyfQPQ6Cknv6a6G1AmfvdUouzYz7L4Ydu/fGftOxv39j42aaRvmsOpCHv5O8QZvMLxSFNAJ+fx8LRwgaGlTMSwJkg9Hi92Z1eFTWzRUnPGaAd8SycSjqqyHqVL2TNyMNgdLxE7TVZDjESRmNcM8kgzWyC4OYRKamSIkWtCbiqaxg5TZjRXHHiQBlfdNk9Wn9Y8AC7fU9mbnvjfrtk5pQx5St3KKezOFExXddGaqWFWDRNzOZQDSG7wu0narhu8kQ/FMtrMj/5iL28djU1JWU5bTGzqpaRN5k4gIRI4agq24w92aUBkUmkWINfZefrpNbWiE/GIVOWkEpbnhT0Nbw0JW1cQ+LBa+CxMBv3TKLy1gs78KAac8GmdBsC5VIUHOq6WqHTakOM7tpdA0paVhUKIxbkUn39LzRLp9MHZgNUkAQuwKOJQN/ap79+3JwDbNx+3NLXEp/Wh8/wBXeP0P7dIxMhvnE5mTa5mlz1f1dol/RNHb4NW2mtelTP1AT8DWEP8IDgkxrhMw7wcmTc6fGD4uMWVHJUEq7KiXX4ECwpIShqh0k20BY+KXGDiZFpoVRUzJkiE+IA/Y+UH4nVJC0dbqj8+zEDR2nJ24m/oLeSLyBnj+IAtAAjA4s0PxJnEwwC4vdYHr4MlUtp23pXiXr6VS2mjQmCQ2RbaWS9vtyw+SznfNwYZKkwtyyU25m2dJmWExMT3fW1JGUhkmBzpHz9ACaAsHvM/XnUeg4ci4BtVJNH0DhUysPue2uMKN80bLxC5GYNtXEWk0u7RbpCrI/kcSJO4ClwESqRoLSIRAbA24bRaskSftoO1alJ3usCg82v2rqREWSbkIWcF8aFRlEqFHGCr/GB8IDGaRLyvLN03a3eB7587mXoVoWBnftxRqvqEiYcdZ+Ey+TMefmuDcOP1uee8u4mwgZj4FGBx2qhbIFdQE/ynLeHnpicztIJucloooJgXDySv4P0CusvyEShp2hzOKhaOH5hBaleli4HUTosAHdVpEL4YSbpkxgsG3dFDOyg1kEFpnLZC2li6Mwl1kjsPHXyfezqyv6TPXVjfv2+sPHs0/vb64HNaJdSEJlRA1d0P7DxadfLkaFNCeRdgg/Yh7ZbLPZIaCIh/UpYjl/TK4hI/UxMpnhWenN8tlRuUwD44TMhaas6cHYFQUO7pX4UVM+7SWY2d7KAWptg3RF4k1NeMa+D9Ht2icWhlpTqu9P8APweK6Myl/IrU9qC6Yowmyga0XI5/YyCn5bAsLKB4cmzmQ6zIs5bCfKSJnNfDJLhxisyAwyk0SIcDtJhoCKkd1sxU079O7EzhbJWbjCNFZzxV42b8URd7kxGYA0tNfsNdp8WZCFvjNtDrzHwE9aN1n8AIv4k/8GLBwDTUVZzQXooT9D9X0Kkp+w94MdYeC1Z53CSzon3AsX957Tlg6xBEwTa0avpUydU5k8qS1m0k0WgdM97linOF/jMEYzMOApMcBo8LSCJRBvEq5paEhr/opCsuTaDugj4rgTn3jxIXJVAzqJ3ylmnyjzxgluEeRHG7DMUJAsO0ojVp7Ynax4TYkuAEx7FsZ3ZnO/EDNynjrVG05xD9k8383SpLJNDDWPUKowH3Lh/K6EkyTSntZtSAGI1k6GpazlTafS4ued5RHAomNThWGwUjLPjJgt/FuxUXW4swZs74uIGcL/jS02bICxXGFnK1oFWpZM7vIeUweGHTBRm/S8ORLf7KT91g7tNq6WzbUBofoyoPlh9UPRzpDAtImFmhk/2In2Fyv28ONc8mt5K/zj3aUmgRBH5THWlKq8qXuA8amIWfJ4Kaz6gmkd2732G7aZlZU76FpbuBhT2TL1A5sccl9a5IAN+wFMhTnQzTfW2yHc39qPc7AjWo9zuABuCn/ZGnGaYfwNtGR7DNq+0/w0+XnpJgrb4BBpiyQyHLDzM+cp2z+/7bePp4LFsHIdVo7xKLP5y6frz4Nz4/bT59t3oK8JvvOjFBmyEQHqqAgltpyfDYqaTB2jU1TgiA7gjWbqd0eY6PpXCWkEs1D8kP5Q6ysoRofNjUrn7qj+WkIhDleLOG+FI4Mq7QS4M9gjfSY+3BWwzwdQM0bE1QlU/qSxiaPLDJ3oRjZiiMeR4gbasszp3/DgZpXbk6cwK3hAm4tMqx+IsmLLXzoHyWUq/gZ1g1q3d3S8E0ed77JtjeCnUvrzypE/e3mX2r3t6VA7ZDvRWUXc2HPKHdi0Q42O6sMzeI6KL5jO8xMd1up2f81Wf58fKb4176vbqD12CbAgwGw9dIlXLm3eXltt2ixqS18FzU8SWOMpzmBvmCIUckrb7XX793kcPlqb1V3r7ORmtV1WijdeHBY4V7YkVnql+DFvPlz/88L4eHZzI2XbsnXi8G9KCiRo9sIwf2X9JhvcQWOIZL3KHht8W3JP4uH6iB57/oc8k3iWE4gx9tVIOmZMzGz5Cyh0ErM4h4edLgaV2bu68CGe6kT1LDl9AsUmzIO8jWpDeMcqjpZFUueJGAtU7gCDgKazZAa8HDdTwp0IDHvQxgQYBX9nNlstns/XtAk76B+zE/+3ZYCHZRRCVVORTfIr6omzx9FofXl1kCItllzI6wYRbjOaemBuOYLJ6uoFUX29bq+2nkhG19Q6qa0EDLxFSf2aWke1tQqn5CsqU5i9hcJPU+2HDWgBK4FXHkexDjc71aUdtLoZqa2NXhNWWVOtfj5Q0mln8Oh449zTFiRekX17Na/qt+qrhom2zveb6nyb1LVzsmFYtJugxXcT6AH80Kmvj4pBi6VUevyAJky4xe0bS7GBQFMPFZVXVAPyDKbrFlpQLcrG2VAbNCcHQBBbQiAij0WIKzVx9acOSSrdxu66V0R7P3UqQIvtIv+o7KfJARU5KgJCoenteMYB11uYRGE3Js832X76A0ULCkypW01qM3wUGcyOIQP5y9Pjn1mbuDUJm3Zdnj7vus+ir0/sb75jVFYrhdXLuoNvKIghm76fpfMUmM7KFGSx9ZORB8voZxvA2AOkF3oqdGGPeYuVbNf8lUt5qYrmLRV2/RXoeYWztJQPKOrrvHycSOTswLy4nGTjUHqg3fyIiX94fvjMn/TxG4iGR04+1isDBVm72ZC2DJZzAgV0KSmQuxVodOTmJRyr2QP2V9Uoo950q1uvd9y3VLdtqbrRWt1V/XlgPAVKNumWyHWzcsqoUzmrRE41UHLRyuo/3/VYl09cxgU0lm+BKgaG5PyS9NScaEr5qXpPMWVVYH7qyDgLQ6N7QtRR5NlKybfId8TZWMbdWhqZ7BneNf+2oOgdwtPFHupqbEQ7rMv3g/c37y7OR8blUpyRahReNRaXZqwUbNklYJC7u7PYwMSbRLaaLenAj4Z+j9zZ1cW5wRIjBoA/sbIxJpMEaMULNPb5Brm3MbDSYpscYrAR4Lcj0BnguqDruy7qx67b7DM9ufH/AMEHHddz0AAA'}
for file,payload in payloads.items():
    target=Path('/content')/file
    decoded=gzip.decompress(base64.b64decode(payload)).decode('utf-8')
    target.write_text(decoded)
    py_compile.compile(str(target),doraise=True)
    print('CHECKED',target,'sha256',hashlib.sha256(target.read_bytes()).hexdigest(),'bytes',len(decoded))
assert 'ns_base_original' not in sys.modules or Path(sys.modules['ns_base_original'].__file__).read_text()==(Path('/content')/'ns_base_original.py').read_text()


In [ ]:
#@title 3. Mandatory model-shape, parameter-count, backward, and metric smoke tests (no dataset test labels)
cmd=[sys.executable,'-u','/content/ns_new_baselines.py','--dataset',str(DATASET),'--output',str(NEW_ROOT),'--original-root',str(ORIGINAL_ROOT),'--seed',str(SEED),'--smoke-only']
subprocess.run(cmd,check=True)


In [ ]:
#@title 4. Run all 16 reference-model fits and seal test until completed (resume after disconnect)
# Default protocol matched to original Navier–Stokes source notebook: 45 epochs, validation every 3.
# This is a large GPU experiment. A session checkpoint prevents data loss.
cmd=[sys.executable,'-u','/content/ns_new_baselines.py',
     '--dataset',str(DATASET),'--output',str(NEW_ROOT),'--original-root',str(ORIGINAL_ROOT),
     '--max-epochs',str(MAX_EPOCHS),'--batch-size',str(BATCH_SIZE),
     '--session-minutes',str(SESSION_MINUTES),'--seed',str(SEED)]
print('RUNNING', ' '.join(cmd),flush=True)
subprocess.run(cmd,check=True)


In [ ]:
#@title 5. Inspect cumulative training/validation, completed test results and comparison figures
from IPython.display import display,Image
for filename in ['validation_selection.csv','new_baselines_test_results.csv','ALL_MODELS_COMPARISON.csv']:
    f=NEW_ROOT/filename
    print('\n',filename, 'found' if f.is_file() else 'not available yet')
    if f.is_file():
        d=pd.read_csv(f)
        fields=[c for c in ['method','N_train','observed_input_frames','parameters','real_scalar_dof','best_epoch','best_validation_full_l2','full_l2_mean','forecast_from_t9_l2_mean','final_l2_mean','full_relative_h1_semi_mean','train_seconds'] if c in d.columns]
        display(d[fields].sort_values(['N_train','method']).reset_index(drop=True))
for filename in ['ALL_MODELS_full_l2_mean.png','ALL_MODELS_truth_predictions_N200_test230.png']:
    f=NEW_ROOT/filename
    if f.is_file():display(Image(filename=str(f)))
